# 구글 드라이브 마운트 및 라이브러리 설정

In [ ]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. 라이브러리 설치 (NumPy 버전 충돌 방지)
!pip install "numpy<2" xgboost shap scikit-learn tensorflow matplotlib seaborn

import os
import numpy as np
import pandas as pd
import random
import tensorflow as tf

# 넘파이 버전 확인
print(f"✅ 현재 NumPy 버전: {np.__version__}")

# 재현성을 위한 시드 고정
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seeds(42)

# 데이터 파일 경로 설정

In [ ]:
import os

# 경로 설정 (본인의 드라이브 경로와 일치하는지 확인!)
base_path = "/content/drive/MyDrive/웨이퍼 이상탐지/"
data_file = os.path.join(base_path, "secom.data")
label_file = os.path.join(base_path, "secom_labels.data")

if os.path.exists(data_file) and os.path.exists(label_file):
    print("✅ 파일 확인 완료!")
else:
    print(f"❌ 에러: {base_path} 경로를 다시 확인해주세요.")

# 데이터 로딩 및 전처리

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    # 1) 데이터 로드
    X_raw = pd.read_csv(data_file, sep=r"\s+", header=None)
    y_raw = pd.read_csv(label_file, sep=r"\s+", header=None)[0].replace(-1, 0)

    # 2) 데이터 분할
    train_idx = int(len(X_raw) * 0.7)
    val_idx = int(len(X_raw) * 0.85)

    X_train_raw, y_train_raw = X_raw.iloc[:train_idx], y_raw.iloc[:train_idx]
    X_val_raw, y_val_raw = X_raw.iloc[train_idx:val_idx], y_raw.iloc[train_idx:val_idx]
    X_test_raw, y_test_raw = X_raw.iloc[val_idx:], y_raw.iloc[val_idx:]

    # 3) 전처리 함수
    def preprocess_data(X_tr, X_v, X_te):
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        # 분산 0인 센서 제거
        variances = X_tr.var()
        keep_cols = variances[variances > 1e-8].index

        X_tr_f = scaler.fit_transform(imputer.fit_transform(X_tr[keep_cols]))
        X_v_f = scaler.transform(imputer.transform(X_v[keep_cols]))
        X_te_f = scaler.transform(imputer.transform(X_te[keep_cols]))

        sensor_names = [f"Sensor_{i}" for i in keep_cols]
        return X_tr_f, X_v_f, X_te_f, sensor_names

    X_tr_s, X_v_s, X_te_s, selected_sensors = preprocess_data(X_train_raw, X_val_raw, X_test_raw)
    print(f"▶ 전처리 성공! 남은 유효 센서: {len(selected_sensors)}개")

except Exception as e:
    print(f"❌ 데이터 처리 에러: {e}")

# LSTM 모델 설계 및 학습

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed

# 1) 시퀀스 생성 함수 (정의를 호출부 위로 이동)
def quick_make_sequences(X, y, n_steps=15):
    X_s, y_s = [], []
    y_vals = y.values if hasattr(y, 'values') else y
    for i in range(len(X) - n_steps + 1):
        X_s.append(X[i:i+n_steps])
        y_s.append(y_vals[i+n_steps-1])
    return np.array(X_s), np.array(y_s)

# 2) 모델 구조 설계
def build_lstm_ae(n_steps, n_features):
    inp = Input(shape=(n_steps, n_features))
    enc = LSTM(64, activation="tanh", return_sequences=False)(inp)
    latent = Dense(16, activation="relu", name="latent_space")(enc)
    dec_rep = RepeatVector(n_steps)(latent)
    dec_lstm = LSTM(64, activation="tanh", return_sequences=True)(dec_rep)
    out = TimeDistributed(Dense(n_features))(dec_lstm)
    model = Model(inp, out)
    model.compile(optimizer="adam", loss="mse")
    return model

# 3) 학습 데이터 준비 및 모델 가동
n_steps = 15
n_features = X_tr_s.shape[1]

# 시퀀스 데이터 생성 (인덱싱 정렬을 위해 y_train_seq도 함께 생성)
X_train_seq, y_train_seq = quick_make_sequences(X_tr_s, y_train_raw, n_steps)
X_val_seq, y_val_seq = quick_make_sequences(X_v_s, y_val_raw, n_steps)
X_test_seq, y_test_seq = quick_make_sequences(X_te_s, y_test_raw, n_steps)

# 정상 샘플만 추출하여 학습
X_tr_normal = X_train_seq[y_train_seq == 0]

autoencoder = build_lstm_ae(n_steps, n_features)

print("▶ 모델 학습 시작... (GPU 가동 중)")
autoencoder.fit(X_tr_normal, X_tr_normal, epochs=50, batch_size=64, validation_split=0.1, verbose=1)

# 이상 탐지 및 결과 리포트

In [ ]:
# 1. 이상 점수 계산 함수
def get_reconstruction_loss(model, X):
    X_pred = model.predict(X, verbose=0)
    return np.mean(np.square(X - X_pred), axis=(1, 2))

# 2. 임계치 설정 및 판정
val_loss = get_reconstruction_loss(autoencoder, X_val_seq)
threshold = np.percentile(val_loss[y_val_seq == 0], 95)

test_scores = get_reconstruction_loss(autoencoder, X_test_seq)
test_preds = (test_scores > threshold).astype(int)
anomaly_indices = np.where(test_preds == 1)[0]

# 3. 리포트 출력
print("\n" + "="*50)
print("        [ SECOM 반도체 공정 이상 감지 리포트 ]        ")
print("="*50)
print(f"▶ 탐지된 이상 샘플: {len(anomaly_indices)}개")
if len(anomaly_indices) > 0:
    print(f"⚠️ 경고: #{anomaly_indices[0]}번 웨이퍼에서 이상 흐름 감지!")
print("="*50)

In [ ]:
import numpy as np
import pandas as pd

# 1. 이상 점수(MSE) 계산 함수 정의
def get_reconstruction_loss(model, X):
    X_pred = model.predict(X, verbose=0)
    # 각 시퀀스별 평균 제곱 오차(MSE)를 계산합니다.
    return np.mean(np.square(X - X_pred), axis=(1, 2))

# 2. 손실 값 계산
val_loss = get_reconstruction_loss(autoencoder, X_val_seq)
test_scores = get_reconstruction_loss(autoencoder, X_test_seq)

# 3. 임계치 설정 (정상 데이터의 95백분위수 기준)
# ADsP나 SQLD 자격증 취득 과정에서 다뤘던 통계적 판단 기준을 적용합니다.
threshold = np.percentile(val_loss[y_val_seq == 0], 95)

# 4. 판정 및 결과 분석
test_preds = (test_scores > threshold).astype(int)
anomaly_indices = np.where(test_preds == 1)[0]
anomaly_scores = test_scores[anomaly_indices]

# 5. 상세 리포트 출력
print("="*60)
print("        🔍 [ SECOM 반도체 공정 이상 탐지 상세 리포트 ]        ")
print("="*60)

# [섹션 1: 전체 통계]
print(f"1. 공정 요약 통계")
print(f"   - 전체 테스트 웨이퍼 수  : {len(X_test_seq)}개")
print(f"   - 정상 판정 웨이퍼 수    : {len(X_test_seq) - len(anomaly_indices)}개")
print(f"   - 이상 감지 웨이퍼 수    : {len(anomaly_indices)}개")
print(f"   - 이상 탐지율(Ratio)     : {(len(anomaly_indices)/len(X_test_seq))*100:.2f}%")
print("-" * 60)

# [섹션 2: 임계치 및 점수 분석]
# MSE(Mean Squared Error)는 모델이 정상 패턴을 얼마나 잘 복원하지 못했는지를 나타냅니다.
print(f"2. 이상 점수(MSE) 분석 기준")
print(f"   - 설정된 임계치(Threshold) : {threshold:.6f}")
print(f"   - 테스트 데이터 평균 MSE   : {np.mean(test_scores):.6f}")
print(f"   - 테스트 데이터 최대 MSE   : {np.max(test_scores):.6f}")
print("-" * 60)

# [섹션 3: 개별 이상 샘플 상세 정보]
print(f"3. 탐지된 이상 샘플 상세 리포트")
if len(anomaly_indices) > 0:
    # 최대 10개까지만 리스트업
    list_limit = 10
    print(f"   [상위 {min(len(anomaly_indices), list_limit)}개 리스트]")
    print(f"   {'Index':<10} | {'Anomaly Score':<15} | {'Severity'}")
    print("-" * 45)

    for i in range(min(len(anomaly_indices), list_limit)):
        idx = anomaly_indices[i]
        score = test_scores[idx]
        # 임계치 대비 얼마나 높은지 심각도를 표시합니다.
        severity = "High" if score > threshold * 2 else "Medium"
        print(f"   #{idx:<9} | {score:<15.6f} | {severity}")

    if len(anomaly_indices) > list_limit:
        print(f"   ... 외 {len(anomaly_indices) - list_limit}개 추가 감지됨")
else:
    print("   ✅ 모든 샘플이 정상 범위 내에 존재합니다.")

print("="*60)
print("   💡 다음 단계 권장: 감지된 인덱스(#)에 대해 SHAP 분석을 수행하세요.")
print("="*60)

In [ ]:
# 1. 학습된 autoencoder 모델에서 'latent_space' 레이어까지를 떼어내어 encoder를 만듭니다.
# (우리가 이전 모델링에서 레이어 이름을 "latent_space"라고 지었기 때문에 가능합니다!)
try:
    encoder = Model(inputs=autoencoder.input, outputs=autoencoder.get_layer("latent_space").output)
    print("✅ 모델로부터 encoder 추출 성공! 이제 유사도 분석이 가능합니다.")
except Exception as e:
    print(f"❌ 에러 발생: {e}")
    print("모델 학습 셀(Cell 4)에서 레이어 이름이 'latent_space'로 되어 있는지 확인해주세요.")

✅ 모델로부터 encoder 추출 성공! 이제 유사도 분석이 가능합니다.


In [ ]:
import random
from sklearn.metrics.pairwise import cosine_similarity

# [1] 과거 불량 사례 DB 구축 (친구의 원본 로직 이식)
def build_defect_database(X_seq, y_seq, encoder_model):
    # 실제 라벨이 1(불량)인 데이터만 골라냅니다. # [cite: 2, 10]
    defect_idx = np.where(y_seq == 1)[0]

    # 불량 데이터들의 특징(Embedding)을 추출합니다.
    embeddings = encoder_model.predict(X_seq[defect_idx], verbose=0)

    # 친구가 적어둔 실제 같은 조치사항 리스트 # [cite: 9]
    resolutions = [
        "장비 가스 라인 및 챔버 압력 조건 점검",
        "냉각수 밸브 교체 및 온도 센서 재보정",
        "웨이퍼 이송 로봇 암(Arm) 위치 영점 조절",
        "플라즈마 발생 장치(RF Generator) 전력 안정화",
        "진공 펌프 오일 교체 및 누기(Leak) 테스트 완료"
    ]

    db = []
    for i, idx in enumerate(defect_idx):
        db.append({
            "case_id": int(idx),
            "embedding": embeddings[i],
            "resolution": random.choice(resolutions) # 실제 DB가 없으므로 무작위 할당 #
        })
    return db

# [2] 리포트 출력 실행부
try:
    # DB 구축
    defect_db = build_defect_database(X_train_seq, y_train_seq, encoder)

    # 첫 번째 이상 샘플(anomaly_indices[0])에 대해 유사 사례 검색
    target_idx = anomaly_indices[0]
    current_sample_seq = X_test_seq[target_idx]

    # 유사도 계산 (Cosine Similarity) #
    current_emb = encoder.predict(current_sample_seq[np.newaxis, ...], verbose=0)[0]
    similar_cases = []
    for case in defect_db:
        sim = float(cosine_similarity(current_emb.reshape(1, -1), case["embedding"].reshape(1, -1))[0][0])
        similar_cases.append((sim, case))

    # 유사도 순으로 정렬 후 상위 3개 추출 #
    similar_cases.sort(key=lambda x: x[0], reverse=True)
    top_3_cases = similar_cases[:3]

    # 상세 리포트 출력 #
    print("\n" + "="*60)
    print("        [ SECOM 공정 센서 이상 감지 상세 리포트 ]        ")
    print("="*60)
    print(f"▶ 대상 샘플: Sample ID #{target_idx}")
    print(f"▶ 이상 점수: {test_scores[target_idx]:.4f} (임계치: {threshold:.4f})")

    print("\n[ 과거 유사 불량 사례 분석 (Cosine Similarity) ]") # [cite: 13]
    for sim, case in top_3_cases:
        print(f"- Case #{case['case_id']} (전체 흐름 유사도: {sim*100:.1f}%)")
        print(f"  └ 당시에 수행된 조치: {case['resolution']}") # [cite: 13]

    print("\n[ AI 최종 권장 사항 ]")
    print(f"현재 발생한 #{target_idx}번 불량은 과거 #{top_3_cases[0][1]['case_id']}번 사례와 가장 유사합니다.")
    print(f"따라서 [{top_3_cases[0][1]['resolution']}]를 최우선으로 점검하십시오.") # [cite: 14]
    print("="*60)

except NameError as e:
    print(f"❌ 변수 누락: {e}. 위에서부터 순서대로 셀을 실행했는지 확인해주세요.")
except Exception as e:
    print(f"❌ 예기치 못한 에러: {e}")

# SHAP를 활용한 불량 원인 분석

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. SHAP 분석용 손실 함수 정의
# SHAP이 각 센서 값을 바꿔가며 이상 점수(MSE)가 어떻게 변하는지 관찰합니다.
def model_loss_wrapper(x_flat):
    # SHAP은 데이터를 2차원으로 입력하므로, 다시 LSTM용 3차원(샘플, 시점, 센서)으로 복원합니다.
    x_reshaped = x_flat.reshape(-1, n_steps, len(selected_sensors))
    reconstructed = autoencoder.predict(x_reshaped, verbose=0)
    # 각 시퀀스의 평균 제곱 오차(MSE)를 반환합니다.
    return np.mean(np.square(x_reshaped - reconstructed), axis=(1, 2))

# 2. 배경 데이터(Background) 설정
# '정상'의 기준을 알려주기 위해 학습 데이터 중 정상 샘플 30개만 추출합니다. (메모리 절약)
background_data = X_train_seq[y_train_seq == 0][:30].reshape(30, -1)
explainer = shap.KernelExplainer(model_loss_wrapper, background_data)

# 3. #111번 웨이퍼(이상 샘플) 분석
# 리포트에서 확인된 첫 번째 이상 인덱스를 타겟으로 잡습니다.
target_idx = anomaly_indices[0]
target_sample = X_test_seq[target_idx].reshape(1, -1)

print(f"▶ #{target_idx}번 웨이퍼의 범인 센서 분석 중... (약 1~2분 소요)")
shap_values = explainer.shap_values(target_sample, nsamples=100) # nsamples로 속도 조절

# 4. 결과 시각화
# 센서 이름과 시점(t)을 결합하여 어떤 타이밍에 문제가 생겼는지 표시합니다.
feature_names_flat = [f"{s}_t{t}" for t in range(n_steps) for s in selected_sensors]
shap_df = pd.Series(shap_values[0], index=feature_names_flat).abs().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
shap_df.head(10).plot(kind='barh', color='salmon')
plt.title(f"Wafer #{target_idx} - Culprit Sensor Top 10")
plt.xlabel("SHAP Value (Contribution to Anomaly Score)")
plt.gca().invert_yaxis()
plt.show()

print(f"▶ 분석 완료! #{target_idx}번의 이상을 유발한 핵심 요인은 '{shap_df.index[0]}'입니다.")